# HomecareCCV - Notebook Colab para ML con outcomes reales

Este notebook reproduce la ruta de investigación **ML con desenlaces reales por cohorte** de HomecareCCV.

La motivación metodológica es importante: el pipeline antiguo entrenaba modelos contra `risk_level`, una etiqueta sintética generada por una regla clínica determinista. Eso producía métricas cercanas a 0.99 porque el modelo reaprendía la regla. En este notebook usamos outcomes reales de los datasets públicos:

- Cohorte `stroke`: target real `stroke`.
- Cohorte `cvd`: target real `cardio`.
- Cohorte `heart_failure`: target real `HeartDisease`.

La regla MEWS/Framingham se conserva como **baseline clínico auditable**, no como etiqueta de entrenamiento.

## 1. Configuración del notebook

Por defecto se ejecuta una versión liviana para clase/demo con tres modelos representativos y pocos bootstraps. Para reproducir los artefactos completos del repositorio, cambia:

```python
RUN_ALL_MODELS = True
BOOTSTRAP_ITERATIONS = 200
```

El único paso manual es subir `kaggle.json` para descargar los tres datasets reales.

In [ ]:
REPO_URL = "https://github.com/cmorregof/homecare.git"
BRANCH = "main"

# Modo liviano para Colab/clase. Cambia a True para correr los 10 modelos.
RUN_ALL_MODELS = False
LIGHTWEIGHT_MODELS = ["logistic_regression", "gradient_boosting", "catboost"]
BOOTSTRAP_ITERATIONS = 50

PROJECT_DIR = "/content/homecare"

## 2. Preparar entorno Colab

Esta celda clona el repositorio, instala dependencias mínimas y deja el directorio de trabajo listo. No se guardan credenciales en el repo.

In [ ]:
!pip -q install kaggle
!rm -rf {PROJECT_DIR}
!git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!pip -q install -r backend/requirements.txt

## 3. Subir `kaggle.json` y descargar datasets reales

Descarga tu token desde Kaggle: `Account > Settings > API > Create New Token`. Luego sube `kaggle.json` cuando Colab lo pida.

Los archivos descargados son:

- `healthcare-dataset-stroke-data.csv`
- `cardio_train.csv`
- `heart.csv`

In [ ]:
from pathlib import Path

try:
    from google.colab import files
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise FileNotFoundError("Debes subir kaggle.json para descargar los datasets reales.")
    Path("/root/.kaggle").mkdir(parents=True, exist_ok=True)
    !cp kaggle.json /root/.kaggle/kaggle.json
    !chmod 600 /root/.kaggle/kaggle.json
except ModuleNotFoundError:
    print("No estás en Colab. Asegúrate de tener ~/.kaggle/kaggle.json configurado localmente.")

!mkdir -p data/mock
!kaggle datasets download fedesoriano/stroke-prediction-dataset -p data/mock/ --unzip
!kaggle datasets download sulianova/cardiovascular-disease-dataset -p data/mock/ --unzip
!kaggle datasets download fedesoriano/heart-failure-prediction -p data/mock/ --unzip
!ls -lh data/mock

## 4. Ejemplo pequeño: cargar cohortes y auditar leakage

Antes de entrenar, inspeccionamos las cohortes y verificamos que el outcome real no quede como feature derivada. Esta es la corrección central del pipeline.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("backend").resolve()))

from data.etl.unify_datasets import load_real_outcome_cohorts
from ml.real_outcomes import prepare_cohort

cohorts = load_real_outcome_cohorts(Path("data/mock"))
for name, frame in cohorts.items():
    prepared = prepare_cohort(name, frame)
    print(f"\nCOHORTE: {name}")
    print(f"Filas: {len(frame):,}")
    print(f"Prevalencia outcome: {frame['outcome'].mean():.3f}")
    print(f"Features usadas: {len(prepared.features)}")
    print("Features removidas por leakage:", prepared.leakage_removed_features)
    print("Features constantes/casi constantes:", prepared.dropped_constant_features)
    assert prepared.leakage_passed

## 5. Ejecutar pipeline completo de evaluación

La suite calcula:

- ROC-AUC y AUC-PR.
- Brier score y calibración.
- Calibración Platt/isotónica si mejora Brier en validación.
- Curva de decisión: modelo vs tratar-todos, tratar-nadie y score-regla.
- Comparación explícita ML vs score-regla MEWS/Framingham.
- Subgrupos por sexo y edad.
- SHAP/top factores.
- Intervalos de confianza bootstrap.

En modo liviano se usan tres modelos representativos. En modo completo se omite `--models` y se entrenan las diez familias.

In [ ]:
models_arg = "" if RUN_ALL_MODELS else "--models " + " ".join(LIGHTWEIGHT_MODELS)
cmd = f"PYTHONPATH=backend python -m ml.real_outcomes --bootstrap-iterations {BOOTSTRAP_ITERATIONS} {models_arg}"
print(cmd)
!{cmd}

## 6. Cargar resultados principales

El JSON generado es el artefacto central de reproducibilidad. Contiene métricas, calibración, decision curve, subgrupos, SHAP/top features y auditorías.

In [ ]:
import json
import pandas as pd
from pathlib import Path

results_path = Path("backend/ml/models/real_outcomes/real_outcome_results.json")
results = json.loads(results_path.read_text())
summary = pd.DataFrame(results["summary"])

cols = [
    "cohort", "rows", "outcome_prevalence", "best_model",
    "test_roc_auc", "test_auc_pr", "test_brier",
    "rule_roc_auc", "rule_auc_pr", "rule_brier",
    "delta_mean_net_benefit_vs_rule", "leakage_passed",
]
summary[cols]

## 7. Sanity checks

Estos checks hacen fallar el notebook si se reintroduce leakage o si falta un artefacto esperado.

In [ ]:
assert results_path.exists(), "No se generó real_outcome_results.json"
assert set(summary["cohort"]) == {"stroke", "cvd", "heart_failure"}
assert summary["leakage_passed"].all(), "Falló la auditoría de leakage"

for cohort, payload in results["cohorts"].items():
    audit = payload["leakage_audit"]
    forbidden = set(audit["removed_target_derived_features"])
    used = set(payload["features_used"])
    assert not forbidden.intersection(used), f"Leakage en {cohort}: {forbidden.intersection(used)}"
    assert payload["best_model"] in payload["models"]

print("Sanity checks OK: resultados, cohortes, leakage y modelos presentes.")

## 8. Comparar ML contra score-regla

El score-regla MEWS/Framingham se evalúa contra el outcome real. Si el ML aporta valor, debería mejorar discriminación y/o utilidad clínica frente a esa regla.

In [ ]:
comparison = summary.assign(
    delta_roc_auc = summary["test_roc_auc"] - summary["rule_roc_auc"],
    delta_auc_pr = summary["test_auc_pr"] - summary["rule_auc_pr"],
    delta_brier = summary["test_brier"] - summary["rule_brier"],
)[[
    "cohort", "best_model",
    "test_roc_auc", "rule_roc_auc", "delta_roc_auc",
    "test_auc_pr", "rule_auc_pr", "delta_auc_pr",
    "test_brier", "rule_brier", "delta_brier",
    "delta_mean_net_benefit_vs_rule",
]]
comparison

## 9. Calibración y curva de decisión

Mostramos las figuras del mejor modelo por cohorte. En investigación clínica no basta con AUC: la calibración y el beneficio neto importan para decidir si una probabilidad puede orientar revisión humana.

In [ ]:
from IPython.display import display, Image

for cohort, payload in results["cohorts"].items():
    best = payload["best_model"]
    figures = payload["models"][best]["figures"]
    print(f"\n{cohort} - {best} - calibración")
    display(Image(filename=figures["calibration"]))
    print(f"{cohort} - {best} - curva de decisión")
    display(Image(filename=figures["decision_curve"]))

## 10. Subgrupos y factores explicativos

Revisamos desempeño por sexo y franjas de edad. También mostramos top factores SHAP/proxy para el caso de mayor probabilidad en test.

In [ ]:
for cohort, payload in results["cohorts"].items():
    best = payload["best_model"]
    model_payload = payload["models"][best]
    print("\nCOHORTE:", cohort, "| MODELO:", best)
    print("Subgrupos por sexo")
    display(pd.DataFrame(model_payload["subgroups"]["sex"]).T)
    print("Subgrupos por edad")
    display(pd.DataFrame(model_payload["subgroups"]["age_band"]).T)
    print("Top factores")
    display(pd.DataFrame(model_payload["shap_top_features"]))

## 11. Interpretación

Las métricas son más bajas que el pipeline antiguo porque ahora el modelo predice desenlaces reales y no una etiqueta determinista. Eso es una señal de mayor validez, no de fracaso.

Lectura esperada al correr todos los modelos:

- Stroke: AUC moderado, AUC-PR bajo por baja prevalencia.
- CVD: AUC alrededor de 0.8, con mejora frente al score-regla.
- Heart failure: AUC más alto, pero con cohorte pequeña; interpretar con intervalos bootstrap.

## 12. Próximos pasos

Este notebook conecta con HomecareCCV de esta forma:

1. Mantiene la estratificación operacional del bot separada de la validación científica.
2. Permite defender por qué el target sintético no debe presentarse como ML clínico validado.
3. Deja una ruta reproducible para comparar nuevos modelos contra outcomes reales.
4. Puede ampliarse con validación externa, datos longitudinales reales y evaluación prospectiva.

No incluye datos privados, credenciales ni datos de pacientes reales.